<a href="https://colab.research.google.com/github/TediBalint/AI-Jegyzetek/blob/master/LLM%20ek%20Finomhangol%C3%A1sa%20DPO%20%C3%A9s%20PPO%20%C3%96sszehasonl%C3%ADt%C3%A1s.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM-ek Finomhangolása: DPO és PPO Összehasonlítás

**Licenc: CC BY-NC-SA 4.0**

Két fő megközelítés az LLM-ek emberi preferenciákhoz igazítására: **PPO** (hagyományos RLHF) és **DPO** (egyszerűbb, RM nélkül).

## PPO vs DPO összehasonlítás

| | PPO (RLHF) | DPO |
|---|-----|-----|
| **Típus** | RL-based | Direct optimization |
| **Reward model** | Szükséges (külön model) | Nem kell |
| **Modell szám** | 4 (policy, ref, RM, value) | 2 (policy, ref) |
| **Komplexitás** | Magas | Alacsony |
| **Stabilitás** | Nehezebb (RL instabilitás) | Könnyebb |
| **Online sampling** | Kell (generálás) | Nem kell (offline) |

## Melyiket válaszd?

- **DPO**: Ha van preference adat, egyszerűbb implementáció kell
- **PPO**: Ha online tanulás kell, vagy a reward model fontosabb

## Tartalomjegyzék
1. PPO (Proximal Policy Optimization)
2. DPO (Direct Preference Optimization)
3. Vizuális összehasonlítás
4. Gyakorlati implementáció (TRL)
5. Variánsok és alternatívák

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

## 1. PPO (Proximal Policy Optimization)

A **PPO** (Schulman et al., 2017) az RLHF standard RL algoritmusa.

### PPO Objective

$$L_{PPO} = \mathbb{E}\left[\min\left(r_t A_t, \text{clip}(r_t, 1-\epsilon, 1+\epsilon) A_t\right)\right]$$

Ahol:
- $r_t = \frac{\pi_\theta(a|s)}{\pi_{\theta_{old}}(a|s)}$ - probability ratio
- $A_t$ - advantage (reward - baseline)
- $\epsilon$ - clipping parameter (~0.2)

### PPO RLHF-ben

1. **Sample**: Generálj válaszokat a policy-val
2. **Score**: Reward model értékeli
3. **Advantage**: $A = R - \beta \cdot KL$ (KL penalty)
4. **Update**: PPO clipped update

### Szükséges komponensek

- **Policy model**: Az LLM amit tanítunk
- **Reference model**: Fagyasztott SFT model (KL anchor)
- **Reward model**: Emberi preferenciákból tanult
- **Value model**: Baseline / advantage estimation

In [ ]:
class PPOTrainer:
    def __init__(self, policy, ref_policy, reward_model, beta=0.1, clip_eps=0.2):
        self.policy = policy
        self.ref_policy = ref_policy  # Frozen reference
        self.reward_model = reward_model
        self.beta = beta
        self.clip_eps = clip_eps

    def compute_loss(self, prompts, responses, old_log_probs):
        # 1. Get rewards
        rewards = self.reward_model(responses)

        # 2. Current log probs
        log_probs = self.policy.log_prob(prompts, responses)
        ref_log_probs = self.ref_policy.log_prob(prompts, responses)

        # 3. KL penalty
        kl = log_probs - ref_log_probs

        # 4. Advantages (reward - KL penalty)
        advantages = rewards - self.beta * kl

        # 5. PPO clipped objective
        ratio = torch.exp(log_probs - old_log_probs)
        clipped = torch.clamp(ratio, 1 - self.clip_eps, 1 + self.clip_eps)

        return -torch.min(ratio * advantages, clipped * advantages).mean()

print("PPO: 4 model kell (policy, ref, reward, value)")

## 2. DPO (Direct Preference Optimization)

A **DPO** (Rafailov et al., 2023) közvetlenül optimalizál preferencia adatokra, **reward model nélkül**.

### Kulcs felismerés

A DPO megmutatja, hogy az RLHF objective átírható zárt formában:

$$L_{DPO} = -\mathbb{E}\left[\log\sigma\left(\beta \log\frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \beta \log\frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}\right)\right]$$

Ahol:
- $y_w$: chosen (winner) response
- $y_l$: rejected (loser) response
- $\pi_\theta$: policy model
- $\pi_{ref}$: reference model
- $\beta$: regularization strength

### Miért működik?

A DPO implicit módon optimalizálja ugyanazt az objective-et mint a PPO, de:
- Nem kell reward model
- Nem kell online sampling
- Egyszerű supervised learning loss!

In [ ]:
def dpo_loss(policy_chosen_logps, policy_rejected_logps,
             ref_chosen_logps, ref_rejected_logps, beta=0.1):
    """
    DPO loss - egyszerűbb mint PPO!
    """
    # Log ratio differences
    chosen_ratio = beta * (policy_chosen_logps - ref_chosen_logps)
    rejected_ratio = beta * (policy_rejected_logps - ref_rejected_logps)

    # Bradley-Terry model
    return -F.logsigmoid(chosen_ratio - rejected_ratio).mean()

# Szimulált log probs
policy_chosen = torch.randn(32)
policy_rejected = torch.randn(32) - 0.5  # Worse
ref_chosen = torch.randn(32)
ref_rejected = torch.randn(32)

loss = dpo_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected)
print(f"DPO loss: {loss.item():.4f}")

In [ ]:
class DPOTrainer:
    def __init__(self, policy, ref_policy, beta=0.1):
        self.policy = policy
        self.ref_policy = ref_policy  # Frozen
        self.beta = beta

    def train_step(self, prompt, chosen, rejected):
        # Get log probs
        pi_chosen = self.policy.log_prob(prompt, chosen)
        pi_rejected = self.policy.log_prob(prompt, rejected)

        with torch.no_grad():
            ref_chosen = self.ref_policy.log_prob(prompt, chosen)
            ref_rejected = self.ref_policy.log_prob(prompt, rejected)

        return dpo_loss(pi_chosen, pi_rejected, ref_chosen, ref_rejected, self.beta)

print("DPO: Csak 2 model kell (policy, ref)!")

## 3. Vizuális összehasonlítás

### Pipeline különbségek

**PPO (online RL)**:
```
Prompt → Generate → Reward Model → Advantage → PPO Update
           ↑                          ↓
           └──────────────────────────┘ (iterate)
```

**DPO (offline optimization)**:
```
Preference Data (chosen, rejected) → DPO Loss → Update
                                        ↑
                                   (supervised!)
```

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# PPO pipeline
ax = axes[0]
ax.axis('off')
ax.set_title('PPO Pipeline', fontweight='bold')

ppo_steps = ['Prompt', 'Generate', 'Reward\nModel', 'PPO\nUpdate']
for i, step in enumerate(ppo_steps):
    ax.add_patch(plt.Rectangle((0.1 + i*0.22, 0.4), 0.18, 0.2,
                               facecolor='lightblue', edgecolor='black'))
    ax.text(0.19 + i*0.22, 0.5, step, ha='center', va='center', fontsize=9)
    if i < 3:
        ax.annotate('', xy=(0.3 + i*0.22, 0.5), xytext=(0.28 + i*0.22, 0.5),
                   arrowprops=dict(arrowstyle='->'))

# DPO pipeline
ax = axes[1]
ax.axis('off')
ax.set_title('DPO Pipeline (egyszerűbb!)', fontweight='bold')

dpo_steps = ['Preference\nData', 'DPO Loss', 'Update']
for i, step in enumerate(dpo_steps):
    ax.add_patch(plt.Rectangle((0.15 + i*0.28, 0.4), 0.22, 0.2,
                               facecolor='lightgreen', edgecolor='black'))
    ax.text(0.26 + i*0.28, 0.5, step, ha='center', va='center', fontsize=9)
    if i < 2:
        ax.annotate('', xy=(0.4 + i*0.28, 0.5), xytext=(0.37 + i*0.28, 0.5),
                   arrowprops=dict(arrowstyle='->'))

plt.tight_layout()
plt.show()

## 4. Gyakorlati implementáció (Hugging Face TRL)

A `trl` library mindkét módszert támogatja.

In [ ]:
print("""
# DPO with TRL
from trl import DPOTrainer, DPOConfig
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b")
ref_model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-2-7b")

config = DPOConfig(
    beta=0.1,
    learning_rate=1e-6,
    per_device_train_batch_size=4,
)

trainer = DPOTrainer(
    model=model,
    ref_model=ref_model,
    args=config,
    train_dataset=preference_data,  # chosen, rejected pairs
    tokenizer=tokenizer,
)

trainer.train()
""")

## 5. Variánsok és alternatívák

A DPO sikere után több variáns jelent meg:

| Módszer | Év | Különbség |
|---------|-----|-----------|
| **DPO** | 2023 | Alap direct preference |
| **IPO** | 2023 | Identity mapping (jobb regularizáció) |
| **KTO** | 2024 | Nincs szükség párokra (csak good/bad) |
| **ORPO** | 2024 | Odds ratio preference |
| **SimPO** | 2024 | Egyszerűsített, nincs ref model |

### KTO (Kahneman-Tversky Optimization)

A KTO előnye: **nem kell pár** (chosen, rejected), elég ha tudjuk egy válasz jó vagy rossz.

In [ ]:
# KTO loss (csak chosen VAGY rejected kell)
def kto_loss(policy_logps, ref_logps, is_chosen, beta=0.1):
    ratio = beta * (policy_logps - ref_logps)

    # Chosen: maximize, Rejected: minimize
    chosen_mask = is_chosen.float()
    rejected_mask = 1 - chosen_mask

    chosen_loss = -F.logsigmoid(ratio) * chosen_mask
    rejected_loss = -F.logsigmoid(-ratio) * rejected_mask

    return (chosen_loss + rejected_loss).mean()

print("KTO: Nem kell párosított adat!")

## Összefoglalás

### PPO vs DPO táblázat

| Szempont | PPO | DPO |
|----------|-----|-----|
| **Modellek** | 4 (policy, ref, RM, value) | 2 (policy, ref) |
| **Reward model** | ✓ Szükséges | ✗ Nem kell |
| **Online sampling** | ✓ Kell | ✗ Offline |
| **Stabilitás** | Nehezebb (RL) | Könnyebb (SL) |
| **Compute** | Magas | Alacsony |
| **Példák** | ChatGPT, Claude | LLaMA 2 Chat, Zephyr |

### Főbb tanulságok

1. **DPO** egyszerűbb alternatíva: supervised loss, nincs RM
2. **PPO** flexibilisebb: online tanulás, reward shaping
3. **KTO** és más variánsok tovább egyszerűsítenek
4. A gyakorlatban **DPO** népszerűbb lett (egyszerűség)

### Mikor melyiket?

| Helyzet | Ajánlás |
|---------|---------|
| Van preference adat | DPO |
| Online tanulás kell | PPO |
| Egyszerű implementáció | DPO |
| Reward shaping kell | PPO |
| Nincs párosított adat | KTO |

### Ajánlott olvasmány

- [DPO Paper](https://arxiv.org/abs/2305.18290): "Direct Preference Optimization"
- [TRL Documentation](https://huggingface.co/docs/trl): HuggingFace library
- [RLHF Paper](https://arxiv.org/abs/2203.02155): "Training language models to follow instructions"

### Zárszó

Az LLM alignment aktívan kutatott terület - új módszerek (ORPO, SimPO, stb.) folyamatosan jelennek meg. A DPO és PPO megértése alapot ad a további technikák megértéséhez!